## Concept 6 — HyDE (Hypothetical Document Embeddings)

### What is it?
There is a fundamental mismatch between how **questions** and **answers** are written.
A question is short and interrogative. An answer doc is long and declarative.
In embedding space, these live far apart — so similarity search struggles.

**HyDE solves this:** instead of embedding the question, generate a *hypothetical answer* first, then embed that.
A hypothetical answer looks like a real answer doc — so they're close in embedding space.

### Pipeline
```
Query
  → LLM generates hypothetical answer (no retrieval — just a plausible answer)
  → Embed the HYPOTHETICAL ANSWER (not the question)
  → Use that embedding for vector search
  → Real matching docs retrieved
  → LLM answers using REAL docs (hypothetical answer is discarded)
```

### Important
The hypothetical answer is only used for retrieval — it's thrown away after.
The final answer comes from the real retrieved documents.

### Limitation (what Concept 7 fixes)
Retrieved chunks are small and precise — but the LLM may lack surrounding context to understand the answer fully.
→ Concept 7 (Parent Document Retriever) retrieves small chunks but returns their larger parent for context.

In [ ]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries

chunks, vectorstore, embeddings, llm, prompt = setup()

### Step 1 — Generate Hypothetical Answer
Ask the LLM to write a plausible answer. It doesn't need to be accurate — it just needs to look like an answer.

In [ ]:
def generate_hypothetical_answer(query):
    hyp_prompt = (
        f"Write a detailed paragraph that would be a good answer to this question.\n"
        f"This is for search indexing purposes only — accuracy is not required, just plausibility.\n\n"
        f"Question: {query}"
    )
    return llm.invoke(hyp_prompt).content.strip()

# See the hypothetical answer
query = TEST_QUERIES[0]
hyp_answer = generate_hypothetical_answer(query)
print(f"Query: {query}\n")
print(f"Hypothetical Answer (used for retrieval only):")
print(hyp_answer)

### Step 2 — Embed Hypothetical Answer and Search
The hypothetical answer embedding is much closer to real answer docs than the question embedding.

In [ ]:
# Embed the hypothetical answer
hyp_embedding = embeddings.embed_query(hyp_answer)

# Search using the hypothetical embedding (not the question embedding)
real_docs = vectorstore.similarity_search_by_vector(hyp_embedding, k=3)

print(f"Retrieved {len(real_docs)} real docs using hypothetical embedding:")
for i, doc in enumerate(real_docs):
    print(f"\n--- Real Doc {i+1} ---")
    print(doc.page_content[:300])

### Step 3 — Compare: Normal Search vs HyDE Search
See the difference in retrieved docs between embedding the question vs the hypothetical answer.

In [ ]:
# Normal search — embed the question directly
normal_docs = vectorstore.similarity_search(query, k=3)

print("=== Normal Search (embed question) ===")
for doc in normal_docs:
    print(f"  - {doc.page_content[:100]}...")

print("\n=== HyDE Search (embed hypothetical answer) ===")
for doc in real_docs:
    print(f"  - {doc.page_content[:100]}...")

### Step 4 — Full HyDE Function

In [ ]:
def hyde_rag(query):
    # Step 1: generate hypothetical answer
    hyp_answer = generate_hypothetical_answer(query)

    # Step 2: embed hypothetical answer and search
    hyp_embedding = embeddings.embed_query(hyp_answer)
    real_docs     = vectorstore.similarity_search_by_vector(hyp_embedding, k=3)

    # Step 3: answer using real docs (hypothetical answer is discarded)
    context  = format_docs(real_docs)
    response = llm.invoke(prompt.format_messages(context=context, question=query))
    return response.content

### Test — Standard Queries

In [ ]:
run_queries(hyde_rag, label="Concept 6 — HyDE")